# Reddit Riddles: LLM Reasoning Evaluation

This notebook evaluates Large Language Models (LLMs) on Reddit riddles.

## Project Overview

The goal is to:
1. Extract problem/solution pairs from Reddit riddle threads
2. Evaluate existing LLMs on these problems
3. Implement proper evaluation metrics for comparing LLM outputs to ground truth

## Setup Instructions

**Before running this notebook:**
1. Create a `.env` file in the `nlp-project` directory with your OpenRouter API key:
   ```
   OPENROUTER_API_KEY=your_openrouter_api_key_here
   ```
2. Install required packages:
   ```bash
   pip install datasets pandas numpy openai python-dotenv sentence-transformers scikit-learn matplotlib seaborn tqdm
   ```

## Dataset
- Source: https://huggingface.co/datasets/sileod/reddit-riddles
- Posts: Riddles with metadata (title, body, flair, etc.)
- Comments: Potential solutions and discussions

## Evaluation Challenge
Comparing LLM outputs to ground truth is non-trivial because:
- Answers can be expressed in different ways
- Some riddles have multiple valid answers
- Format and style may vary
- Semantic equivalence matters more than exact match


In [2]:
# Import required libraries
from datasets import load_dataset
import pandas as pd
import numpy as np
import re
import os
import json
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Tuple
from tqdm import tqdm
import concurrent.futures

# LLM and evaluation
from openai import OpenAI
from dotenv import load_dotenv
import sentence_transformers
from sklearn.metrics.pairwise import cosine_similarity

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Load environment variables
load_dotenv()

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 150)

# Set matplotlib style
try:
    plt.style.use('seaborn-v0_8')
except:
    try:
        plt.style.use('seaborn')
    except:
        plt.style.use('default')

print("Libraries imported successfully.")


2025-11-11 13:30:58.820029: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-11 13:30:58.886919: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-11 13:31:00.093250: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Libraries imported successfully.


In [3]:
# Setup OpenAI client for OpenRouter
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY not found in environment variables. Please set it in .env file.")

# Initialize OpenAI client with OpenRouter
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# Initialize sentence transformer for semantic similarity evaluation
print("Loading sentence transformer model for evaluation...")
similarity_model = sentence_transformers.SentenceTransformer('all-MiniLM-L6-v2')
print("Setup complete.")


Loading sentence transformer model for evaluation...
Setup complete.


## Step 1: Load and Explore Dataset


In [4]:
def load_reddit_riddles_dataset():
    '''
    Loads the Reddit riddles dataset from HuggingFace.
    
    :return: Tuple of (posts DataFrame, comments DataFrame)
    :type: Tuple[pd.DataFrame, pd.DataFrame]
    '''
    print("Loading datasets...")
    posts = load_dataset("sileod/reddit-riddles", "submissions")["train"]
    comments = load_dataset("sileod/reddit-riddles", "comments")["train"]
    
    print(f"Loaded {len(posts)} posts and {len(comments)} comments")
    
    # Convert to pandas DataFrames
    posts_df = posts.to_pandas()
    comments_df = comments.to_pandas()
    
    return posts_df, comments_df

# Load datasets
posts_df, comments_df = load_reddit_riddles_dataset()

print(f"\nPosts columns: {posts_df.columns.tolist()}")
print(f"Comments columns: {comments_df.columns.tolist()}")


Loading datasets...
Loaded 40749 posts and 429164 comments

Posts columns: ['author', 'created_utc', 'is_self', 'link_flair_text', 'num_comments', 'permalink', 'score', 'selftext', 'subreddit', 'title', 'url', 'upvote_ratio', 'post_id']
Comments columns: ['subreddit', 'link_id', 'id', 'body', 'score', 'author', 'created_utc', 'parent_id', 'permalink']


In [5]:
# Basic statistics
print("=== DATASET STATISTICS ===")
print(f"Total posts: {len(posts_df)}")
print(f"Solved posts: {(posts_df['link_flair_text'] == 'Solved').sum()}")
print(f"Unsolved posts: {(posts_df['link_flair_text'] == 'Unsolved').sum()}")
print(f"Self posts (text only): {posts_df['is_self'].sum()}")
print(f"External link posts: {(~posts_df['is_self']).sum()}")
print(f"\nTotal comments: {len(comments_df)}")
print(f"Comments with positive scores: {(comments_df['score'] > 0).sum()}")
print(f"Average comment score: {comments_df['score'].mean():.2f}")


=== DATASET STATISTICS ===
Total posts: 40749
Solved posts: 11579
Unsolved posts: 3290
Self posts (text only): 38830
External link posts: 1919

Total comments: 429164
Comments with positive scores: 409009
Average comment score: 2.80


## Step 2: Extract Problem/Solution Pairs


## Performance Optimization for LLM Strategy

The LLM-based strategy now uses **parallel processing** with `ThreadPoolExecutor` to make multiple API calls concurrently. This significantly speeds up processing:

- **Sequential**: ~1.5 seconds per post (50 posts = ~75 seconds)
- **Parallel (10 workers)**: ~5-10 seconds total for 50 posts (5-15x faster!)

**Adjust `max_workers` parameter based on your API rate limits:**
- Lower (5-10): Safer for API rate limits, still much faster than sequential
- Higher (20-50): Faster but may hit rate limits or get throttled
- Default: 10 workers (good balance)


In [6]:
def extract_post_id(link_id):
    '''
    Extracts post_id from link_id format 't3_xxxxx'.
    
    Reddit uses prefixes: 't3_' for posts, 't1_' for comments.
    
    :param link_id: Link ID with prefix (e.g., 't3_xxxxx')
    :type link_id: str
    :return: Post ID without prefix, or None if invalid
    :type: Optional[str]
    '''
    if pd.isna(link_id):
        return None
    if isinstance(link_id, str) and link_id.startswith('t3_'):
        return link_id[3:]  # Remove 't3_' prefix
    return link_id


def filter_solved_posts(posts_df, min_score=5, min_text_length=10):
    '''
    Filters posts to only include solved, text-based riddles.
    
    :param posts_df: DataFrame with posts data
    :param min_score: Minimum post score to include (default = 5)
    :param min_text_length: Minimum text length in characters (default = 10)
    :type posts_df: pd.DataFrame
    :type min_score: int
    :type min_text_length: int
    :return: Filtered DataFrame with solved posts
    :type: pd.DataFrame
    '''
    # Filter for solved, self-posts with text
    solved_posts = posts_df[
        (posts_df['link_flair_text'] == "Solved") &
        (posts_df['is_self'] == True) &
        (posts_df['selftext'].notna()) &
        (posts_df['selftext'].str.len() > 0) &
        (posts_df['score'] > min_score)
    ].copy()
    
    # Combine title and selftext to create full riddle text
    solved_posts['riddle_text'] = (
        solved_posts['title'].fillna('') + '\n\n' + solved_posts['selftext'].fillna('')
    ).str.strip()
    
    # Remove very short posts
    solved_posts = solved_posts[solved_posts['riddle_text'].str.len() >= min_text_length].copy()
    
    return solved_posts


def prepare_comments(comments_df):
    '''
    Prepares comments for matching with posts.
    
    Extracts post_id from link_id and identifies top-level comments
    (direct replies to posts vs nested comments).
    
    :param comments_df: DataFrame with comments data
    :type comments_df: pd.DataFrame
    :return: Processed DataFrame with post_id and is_top_level columns
    :type: pd.DataFrame
    '''
    # Clean comments: remove deleted/removed
    valid_comments = comments_df[
        (comments_df['body'].notna()) &
        (~comments_df['body'].str.lower().isin(['[deleted]', '[removed]', ''])) &
        (comments_df['body'].str.len() >= 1)
    ].copy()
    
    # Extract post_id from link_id
    valid_comments['post_id'] = valid_comments['link_id'].apply(extract_post_id)
    
    # Identify top-level comments (direct replies to post)
    # Top-level: parent_id starts with 't3_' (post)
    # Nested: parent_id starts with 't1_' (comment)
    valid_comments['is_top_level'] = (
        valid_comments['parent_id'].str.startswith('t3_') 
        if valid_comments['parent_id'].notna().any() 
        else pd.Series([False] * len(valid_comments))
    )
    
    return valid_comments


def select_best_answer(comments_for_post, use_top_level_only=True):
    '''
    Selects the best answer from comments for a given post.
    
    Strategy: Highest positive score among top-level comments (or all comments).
    
    :param comments_for_post: DataFrame with comments for a specific post
    :param use_top_level_only: If True, only consider top-level comments (default = True)
    :type comments_for_post: pd.DataFrame
    :type use_top_level_only: bool
    :return: Best comment as Series, or None if no valid answer found
    :type: Optional[pd.Series]
    '''
    if len(comments_for_post) == 0:
        return None
    
    # Filter for top-level comments if requested
    if use_top_level_only:
        comments_for_post = comments_for_post[comments_for_post['is_top_level']]
        if len(comments_for_post) == 0:
            return None
    
    # Filter for positive scores only
    positive_comments = comments_for_post[comments_for_post['score'] > 0]
    if len(positive_comments) == 0:
        return None
    
    # Get comment with highest score
    best_comment = positive_comments.loc[positive_comments['score'].idxmax()]
    return best_comment


def select_best_answer_with_llm(riddle_text, comments_for_post, use_top_level_only=True, 
                                  model="openai/gpt-4o-mini", max_comments=30):
    '''
    Uses an LLM to select the best answer from comments for a given riddle.
    
    The LLM evaluates all comments and can reject all of them if none are satisfactory.
    
    :param riddle_text: The riddle text (problem)
    :param comments_for_post: DataFrame with comments for a specific post
    :param use_top_level_only: If True, only consider top-level comments (default = True)
    :param model: Model identifier for OpenRouter (default = "openai/gpt-4o-mini")
    :param max_comments: Maximum number of comments to evaluate (default = 30)
    :type riddle_text: str
    :type comments_for_post: pd.DataFrame
    :type use_top_level_only: bool
    :type model: str
    :type max_comments: int
    :return: Best comment as Series, or None if no valid answer found or LLM rejects all
    :type: Optional[pd.Series]
    '''
    if len(comments_for_post) == 0:
        return None
    
    # Filter for top-level comments if requested
    if use_top_level_only:
        comments_for_post = comments_for_post[comments_for_post['is_top_level']]
        if len(comments_for_post) == 0:
            return None
    
    # Limit number of comments to avoid token limits (prioritize by score)
    comments_to_evaluate = comments_for_post.nlargest(max_comments, 'score')
    
    if len(comments_to_evaluate) == 0:
        return None
    
    # Format comments for LLM
    comments_list = []
    for idx, (comment_idx, comment) in enumerate(comments_to_evaluate.iterrows()):
        comment_text = comment['body'].strip()
        comment_score = comment['score']
        comments_list.append(f"Comment {idx + 1} (Score: {comment_score}): {comment_text}")
    
    comments_text = "\n\n".join(comments_list)
    
    # Create prompt for LLM
    prompt = f"""You are evaluating potential answers to a riddle. Your task is to identify which comment best answers the riddle, or reject all comments if none provide a satisfactory solution.

RIDDLE:
{riddle_text}

POTENTIAL ANSWERS:
{comments_text}

Please analyze each comment and determine which one is the best answer to the riddle. Consider:
1. Does it directly answer the riddle?
2. Is it clear and specific?
3. Does it make logical sense?
4. Is it a complete answer?

If you find a good answer, respond with ONLY the number of the comment (e.g., "1" or "2" or "3", etc.).
If NONE of the comments provide a satisfactory answer to the riddle, respond with "NONE" or "REJECT".

Respond with ONLY the number or "NONE"/"REJECT". Do not include any other text."""

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.1,  # Low temperature for consistent results
            max_tokens=50
        )
        
        # Extract the response
        answer_text = response.choices[0].message.content.strip().upper()
        
        # Check if LLM rejected all comments
        if "NONE" in answer_text or "REJECT" in answer_text:
            return None
        
        # Extract comment number from response
        match = re.search(r'\d+', answer_text)
        if match:
            comment_num = int(match.group()) - 1  # Convert to 0-based index
            if 0 <= comment_num < len(comments_to_evaluate):
                # Return the selected comment
                selected_comment = comments_to_evaluate.iloc[comment_num]
                return selected_comment
        
        # If parsing fails, return None (reject)
        return None
        
    except Exception as e:
        print(f"LLM error in select_best_answer_with_llm: {e}")
        # Fallback to highest score strategy
        positive_comments = comments_for_post[comments_for_post['score'] > 0]
        if len(positive_comments) > 0:
            return positive_comments.loc[positive_comments['score'].idxmax()]
        return None


In [7]:
def process_single_post(post_data, comments_by_post, use_top_level_only, strategy, llm_model):
    '''
    Processes a single post to extract problem/solution pair.
    
    Helper function for parallel processing. Uses select_best_answer and select_best_answer_with_llm
    from the global scope (these functions must be defined before this function).
    
    :param post_data: Tuple of (post_id, post_row) where post_row is a Series
    :param comments_by_post: GroupBy object with comments grouped by post_id
    :param use_top_level_only: If True, only consider top-level comments
    :param strategy: Strategy for selecting best answer
    :param llm_model: Model identifier for OpenRouter
    :type post_data: Tuple[str, pd.Series]
    :type comments_by_post: pd.core.groupby.DataFrameGroupBy
    :type use_top_level_only: bool
    :type strategy: str
    :type llm_model: str
    :return: Dictionary with problem/solution pair or None, and rejection status
    :type: Tuple[Optional[Dict], bool]
    '''
    post_id, post_row = post_data
    
    if post_id not in comments_by_post.groups:
        return None, False
    
    comments_for_post = comments_by_post.get_group(post_id)
    
    # Select best answer based on strategy
    # Note: select_best_answer and select_best_answer_with_llm are used from global scope
    if strategy == "llm_based":
        best_answer = select_best_answer_with_llm(
            post_row['riddle_text'], 
            comments_for_post, 
            use_top_level_only=use_top_level_only,
            model=llm_model
        )
        rejected = (best_answer is None)
    else:  # score_based (default)
        best_answer = select_best_answer(comments_for_post, use_top_level_only=use_top_level_only)
        rejected = False
    
    if best_answer is not None:
        # Clean answer text (remove Reddit spoiler tags)
        answer_text = best_answer['body']
        answer_text = re.sub(r'&gt;!', '', answer_text)
        answer_text = re.sub(r'!&lt;', '', answer_text)
        answer_text = re.sub(r'&lt;', '', answer_text)
        answer_text = re.sub(r'&gt;', '', answer_text)
        answer_text = re.sub(r'\[/?spoiler\]', '', answer_text, flags=re.IGNORECASE)
        answer_text = re.sub(r'\(?\s*spoiler\s*\)?', '', answer_text, flags=re.IGNORECASE)
        answer_text = re.sub(r'\*\*?spoiler\*\*?', '', answer_text, flags=re.IGNORECASE)
        answer_text = re.sub(r'(?i)\b(answer|solution|riddle answer|correct answer|my guess|i think it\'s)\b[:\-]?', '', answer_text)
        answer_text = re.sub(r'(?i)\bthe answer is\b[:\-]?', '', answer_text)
        answer_text = re.sub(r'(?i)\bthat would be\b[:\-]?', '', answer_text)
        answer_text = re.sub(r'\*{1,2}', '', answer_text)
        answer_text = re.sub(r'[_~>`#]', '', answer_text)
        answer_text = re.sub(r'\s+', ' ', answer_text)
        answer_text = answer_text.strip()
        
        return {
            'post_id': post_id,
            'problem': post_row['riddle_text'],
            'solution': answer_text,
            'solution_score': best_answer['score'],
            'solution_is_top_level': best_answer['is_top_level'],
            'num_comments': len(comments_for_post),
            'title': post_row['title'],
            'strategy': strategy
        }, rejected
    
    return None, rejected


def extract_problem_solution_pairs(posts_df, comments_df, use_top_level_only=True, 
                                     strategy="score_based", llm_model="openai/gpt-4o-mini",
                                     sample_size=None, max_workers=10):
    '''
    Extracts problem/solution pairs from posts and comments.
    
    Supports multiple strategies for selecting the best answer:
    - "score_based": Uses highest scoring comment (default)
    - "llm_based": Uses LLM to evaluate and select best answer, can reject if none are good
    
    When using "llm_based" strategy, processes posts in parallel using ThreadPoolExecutor
    for much faster execution.
    
    :param posts_df: DataFrame with filtered solved posts
    :param comments_df: DataFrame with processed comments
    :param use_top_level_only: If True, only consider top-level comments (default = True)
    :param strategy: Strategy for selecting best answer - "score_based" or "llm_based" (default = "score_based")
    :param llm_model: Model identifier for OpenRouter when using "llm_based" strategy (default = "openai/gpt-4o-mini")
    :param sample_size: Number of posts to sample (None = process all posts)
    :param max_workers: Maximum number of parallel workers for LLM strategy (default = 10)
                        Adjust based on API rate limits. Higher = faster but may hit rate limits.
    :type posts_df: pd.DataFrame
    :type comments_df: pd.DataFrame
    :type use_top_level_only: bool
    :type strategy: str
    :type llm_model: str
    :type sample_size: Optional[int]
    :type max_workers: int
    :return: DataFrame with problem/solution pairs
    :type: pd.DataFrame
    '''
    # Filter solved posts
    solved_posts = filter_solved_posts(posts_df)
    print(f"Filtered to {len(solved_posts)} solved self-posts")
    
    # Apply limited sample if requested
    if sample_size is not None and sample_size < len(solved_posts):
        solved_posts = solved_posts.head(sample_size).copy()
        print(f"Processing {len(solved_posts)} posts")
    
    # Prepare comments
    valid_comments = prepare_comments(comments_df)
    print(f"Valid comments: {len(valid_comments)}")
    
    # Group comments by post_id
    comments_by_post = valid_comments.groupby('post_id')
    
    # Extract problem/solution pairs
    problem_solution_pairs = []
    rejected_posts = 0
    
    print(f"Using strategy: {strategy}")
    
    # Prepare post data as list of tuples
    post_data_list = [(row['post_id'], row) for _, row in solved_posts.iterrows()]
    
    if strategy == "llm_based":
        # Use parallel processing for LLM strategy
        print(f"Using parallel processing with {max_workers} workers...")
        print("Note: Adjust max_workers based on your API rate limits")
        
        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
            # Submit all tasks
            future_to_post = {
                executor.submit(
                    process_single_post, 
                    post_data, 
                    comments_by_post, 
                    use_top_level_only, 
                    strategy, 
                    llm_model
                ): post_data[0] 
                for post_data in post_data_list
            }
            
            # Process completed tasks with progress bar
            with tqdm(total=len(post_data_list), desc="Extracting pairs (parallel)") as pbar:
                for future in concurrent.futures.as_completed(future_to_post):
                    post_id = future_to_post[future]
                    try:
                        result, rejected = future.result()
                        if result is not None:
                            problem_solution_pairs.append(result)
                        if rejected:
                            rejected_posts += 1
                    except Exception as e:
                        print(f"\nError processing post {post_id}: {e}")
                    pbar.update(1)
    else:
        # Sequential processing for score-based strategy (fast enough)
        for post_data in tqdm(post_data_list, desc="Extracting pairs"):
            result, rejected = process_single_post(
                post_data, 
                comments_by_post, 
                use_top_level_only, 
                strategy, 
                llm_model
            )
            if result is not None:
                problem_solution_pairs.append(result)
            if rejected:
                rejected_posts += 1
    
    pairs_df = pd.DataFrame(problem_solution_pairs)
    print(f"\nExtracted {len(pairs_df)} problem/solution pairs")
    if strategy == "llm_based" and rejected_posts > 0:
        print(f"LLM rejected {rejected_posts} posts (no satisfactory answer found)")
        if len(post_data_list) > 0:
            print(f"Rejection rate: {rejected_posts / len(post_data_list) * 100:.1f}%")
    
    return pairs_df

## Step 3: LLM Evaluation Functions (Parallel Processing)

In [8]:
def create_riddle_prompt(problem_text):
    '''
    Creates a prompt for LLM to solve a riddle.
    
    :param problem_text: The riddle text (problem)
    :type problem_text: str
    :return: Formatted prompt string
    :type: str
    '''
    prompt = f"""Solve the following riddle. Provide your answer clearly and concisely.

    RIDDLE:
    {problem_text}

    ANSWER:"""
    return prompt


def solve_riddle_with_llm(problem_text, model="openai/gpt-4o-mini", temperature=0.1, max_tokens=200):
    '''
    Uses an LLM to solve a riddle.
    
    :param problem_text: The riddle text (problem)
    :param model: Model identifier for OpenRouter (default = "openai/gpt-4o-mini")
    :param temperature: Sampling temperature (default = 0.1)
    :param max_tokens: Maximum tokens in response (default = 200)
    :type problem_text: str
    :type model: str
    :type temperature: float
    :type max_tokens: int
    :return: LLM's answer to the riddle, or None if error
    :type: Optional[str]
    '''
    prompt = create_riddle_prompt(problem_text)
    
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )
        
        answer = response.choices[0].message.content.strip()
        return answer
    except Exception as e:
        print(f"Error calling LLM: {e}")
        return None


def normalize_text(text):
    '''
    Normalizes text for comparison by removing punctuation and converting to lowercase.
    
    :param text: Text to normalize
    :type text: str
    :return: Normalized text
    :type: str
    '''
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    # Remove extra whitespace
    text = ' '.join(text.split())
    return text


def exact_match_score(predicted_answer, ground_truth_answer):
    '''
    Computes exact match score between predicted and ground truth answers.
    
    :param predicted_answer: LLM's predicted answer
    :param ground_truth_answer: Ground truth answer from dataset
    :type predicted_answer: str
    :type ground_truth_answer: str
    :return: 1.0 if exact match (after normalization), 0.0 otherwise
    :type: float
    '''
    pred_norm = normalize_text(predicted_answer)
    truth_norm = normalize_text(ground_truth_answer)
    
    return 1.0 if pred_norm == truth_norm else 0.0


def semantic_similarity_score(predicted_answer, ground_truth_answer, model=None):
    '''
    Computes semantic similarity score between predicted and ground truth answers.
    
    Uses sentence transformers to compute cosine similarity of embeddings.
    
    :param predicted_answer: LLM's predicted answer
    :param ground_truth_answer: Ground truth answer from dataset
    :param model: Sentence transformer model (default = None, uses global model)
    :type predicted_answer: str
    :type ground_truth_answer: str
    :type model: Optional[sentence_transformers.SentenceTransformer]
    :return: Cosine similarity score between 0 and 1
    :type: float
    '''
    if model is None:
        model = similarity_model
    
    # Get embeddings
    embeddings = model.encode([predicted_answer, ground_truth_answer])
    
    # Compute cosine similarity
    similarity = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
    
    return float(similarity)


def contains_keywords(predicted_answer, ground_truth_answer):
    '''
    Checks if predicted answer contains key words from ground truth.
    
    :param predicted_answer: LLM's predicted answer
    :param ground_truth_answer: Ground truth answer from dataset
    :type predicted_answer: str
    :type ground_truth_answer: str
    :return: Ratio of ground truth words found in predicted answer
    :type: float
    '''
    pred_words = set(normalize_text(predicted_answer).split())
    truth_words = set(normalize_text(ground_truth_answer).split())
    
    if len(truth_words) == 0:
        return 0.0
    
    # Remove common stop words for better matching
    stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by'}
    truth_words = truth_words - stop_words
    
    if len(truth_words) == 0:
        return 0.0
    
    matched_words = pred_words.intersection(truth_words)
    return len(matched_words) / len(truth_words)


def evaluate_answer(predicted_answer, ground_truth_answer, model=None):
    '''
    Evaluates a predicted answer against ground truth using multiple metrics.
    
    :param predicted_answer: LLM's predicted answer
    :param ground_truth_answer: Ground truth answer from dataset
    :param model: Sentence transformer model for semantic similarity (default = None)
    :type predicted_answer: str
    :type ground_truth_answer: str
    :type model: Optional[sentence_transformers.SentenceTransformer]
    :return: Dictionary with evaluation metrics
    :type: Dict[str, float]
    '''
    if predicted_answer is None:
        predicted_answer = ""
    
    metrics = {
        'exact_match': exact_match_score(predicted_answer, ground_truth_answer),
        'semantic_similarity': semantic_similarity_score(predicted_answer, ground_truth_answer, model),
        'keyword_overlap': contains_keywords(predicted_answer, ground_truth_answer)
    }
    
    return metrics

print("LLM evaluation functions defined.")


LLM evaluation functions defined.


In [9]:
def process_single_riddle_evaluation(row_data, model):
    '''
    Processes a single riddle evaluation.
    
    Helper function for parallel processing. Uses solve_riddle_with_llm and evaluate_answer
    from the global scope (these functions must be defined before this function).
    
    :param row_data: Tuple of (index, row) where row is a Series with problem/solution data
    :param model: Model identifier for OpenRouter
    :type row_data: Tuple[int, pd.Series]
    :type model: str
    :return: Dictionary with evaluation results, or None if error
    :type: Optional[Dict]
    '''
    idx, row = row_data
    
    try:
        problem = row['problem']
        ground_truth = row['solution']
        
        # Get LLM's answer
        predicted_answer = solve_riddle_with_llm(problem, model=model)
        
        # Evaluate answer
        metrics = evaluate_answer(predicted_answer, ground_truth)
        
        # Store results
        result = {
            'post_id': row['post_id'],
            'problem': problem,
            'ground_truth': ground_truth,
            'predicted_answer': predicted_answer,
            'model': model,
            'exact_match': metrics['exact_match'],
            'semantic_similarity': metrics['semantic_similarity'],
            'keyword_overlap': metrics['keyword_overlap'],
            'solution_score': row['solution_score']
        }
        return result
    except Exception as e:
        print(f"\nError evaluating riddle {row.get('post_id', idx)}: {e}")
        return None


def evaluate_llm_on_riddles(pairs_df, model="openai/gpt-4o-mini", max_samples=None, 
                             save_results=True, max_workers=10):
    '''
    Evaluates an LLM on a set of riddle problem/solution pairs.
    
    Uses parallel processing with ThreadPoolExecutor for faster execution (5-15x speedup).
    
    :param pairs_df: DataFrame with problem/solution pairs
    :param model: Model identifier for OpenRouter (default = "openai/gpt-4o-mini")
    :param max_samples: Maximum number of samples to evaluate (default = None, evaluates all)
    :param save_results: If True, save results to JSON file (default = True)
    :param max_workers: Maximum number of parallel workers (default = 10)
                        Adjust based on API rate limits. Higher = faster but may hit rate limits.
    :type pairs_df: pd.DataFrame
    :type model: str
    :type max_samples: Optional[int]
    :type save_results: bool
    :type max_workers: int
    :return: DataFrame with evaluation results
    :type: pd.DataFrame
    '''
    # Limit samples if specified
    if max_samples is not None:
        pairs_df = pairs_df.head(max_samples).copy()
    
    results = []
    
    print(f"Evaluating {model} on {len(pairs_df)} riddles...")
    print(f"Using parallel processing with {max_workers} workers...")
    print("Note: Adjust max_workers based on your API rate limits")
    
    # Prepare row data as list of tuples
    row_data_list = [(idx, row) for idx, row in pairs_df.iterrows()]
    
    # Use parallel processing
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        future_to_row = {
            executor.submit(process_single_riddle_evaluation, row_data, model): row_data[0]
            for row_data in row_data_list
        }
        
        # Process completed tasks with progress bar
        with tqdm(total=len(row_data_list), desc=f"Evaluating {model}") as pbar:
            for future in concurrent.futures.as_completed(future_to_row):
                idx = future_to_row[future]
                try:
                    result = future.result()
                    if result is not None:
                        results.append(result)
                except Exception as e:
                    print(f"\nError processing riddle at index {idx}: {e}")
                pbar.update(1)
    
    results_df = pd.DataFrame(results)
    
    # Save results if requested
    if save_results:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_file = f"evaluation_results_{model.replace('/', '_')}_{timestamp}.json"
        results_df.to_json(output_file, orient='records', indent=2)
        print(f"\nResults saved to {output_file}")
    
    return results_df

print("Evaluation function with parallel processing defined.")


Evaluation function with parallel processing defined.


In [10]:
def analyze_evaluation_results(results_df):
    '''
    Analyzes evaluation results and computes summary statistics.
    
    :param results_df: DataFrame with evaluation results
    :type results_df: pd.DataFrame
    :return: Dictionary with summary statistics
    :type: Dict[str, float]
    '''
    stats = {
        'total_samples': len(results_df),
        'exact_match_rate': results_df['exact_match'].mean(),
        'avg_semantic_similarity': results_df['semantic_similarity'].mean(),
        'avg_keyword_overlap': results_df['keyword_overlap'].mean(),
        'high_similarity_rate': (results_df['semantic_similarity'] >= 0.7).mean(),
        'perfect_matches': results_df['exact_match'].sum()
    }
    
    return stats


def display_evaluation_summary(results_df):
    '''
    Displays a summary of evaluation results.
    
    :param results_df: DataFrame with evaluation results
    :type results_df: pd.DataFrame
    '''
    stats = analyze_evaluation_results(results_df)
    
    print("="*80)
    print("EVALUATION SUMMARY")
    print("="*80)
    print(f"Total samples: {stats['total_samples']}")
    print(f"Exact match rate: {stats['exact_match_rate']:.2%}")
    print(f"Average semantic similarity: {stats['avg_semantic_similarity']:.3f}")
    print(f"Average keyword overlap: {stats['avg_keyword_overlap']:.3f}")
    print(f"High similarity rate (>= 0.7): {stats['high_similarity_rate']:.2%}")
    print(f"Perfect matches: {stats['perfect_matches']}")
    print("="*80)


def display_sample_results(results_df, n=5, show_best=True):
    '''
    Displays sample evaluation results.
    
    :param results_df: DataFrame with evaluation results
    :param n: Number of samples to display (default = 5)
    :param show_best: If True, show best results; if False, show worst (default = True)
    :type results_df: pd.DataFrame
    :type n: int
    :type show_best: bool
    '''
    # Sort by semantic similarity
    sorted_df = results_df.sort_values('semantic_similarity', ascending=not show_best)
    samples = sorted_df.head(n)
    
    print(f"\n{'='*80}")
    print(f"SAMPLE RESULTS ({'BEST' if show_best else 'WORST'} {n})")
    print(f"{'='*80}\n")
    
    for idx, row in samples.iterrows():
        print(f"Post ID: {row['post_id']}")
        print(f"Semantic Similarity: {row['semantic_similarity']:.3f}")
        print(f"Exact Match: {'Yes' if row['exact_match'] else 'No'}")
        print(f"\nProblem (first 150 chars): {row['problem'][:150]}...")
        print(f"\nGround Truth: {row['ground_truth']}")
        print(f"Predicted: {row['predicted_answer']}")
        print(f"\n{'-'*80}\n")


In [11]:
def plot_evaluation_metrics(results_df):
    '''
    Creates visualizations of evaluation metrics.
    
    :param results_df: DataFrame with evaluation results
    :type results_df: pd.DataFrame
    '''
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('LLM Evaluation Results on Reddit Riddles', fontsize=16, fontweight='bold')
    
    # 1. Semantic similarity distribution
    ax1 = axes[0, 0]
    ax1.hist(results_df['semantic_similarity'], bins=30, alpha=0.7, edgecolor='black')
    ax1.axvline(results_df['semantic_similarity'].mean(), color='red', linestyle='--', 
                label=f"Mean: {results_df['semantic_similarity'].mean():.3f}")
    ax1.set_xlabel('Semantic Similarity')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Semantic Similarity Distribution')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Keyword overlap distribution
    ax2 = axes[0, 1]
    ax2.hist(results_df['keyword_overlap'], bins=30, alpha=0.7, edgecolor='black', color='green')
    ax2.axvline(results_df['keyword_overlap'].mean(), color='red', linestyle='--', 
                label=f"Mean: {results_df['keyword_overlap'].mean():.3f}")
    ax2.set_xlabel('Keyword Overlap')
    ax2.set_ylabel('Frequency')
    ax2.set_title('Keyword Overlap Distribution')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. Exact match vs semantic similarity
    ax3 = axes[1, 0]
    exact_match = results_df[results_df['exact_match'] == 1.0]
    no_exact_match = results_df[results_df['exact_match'] == 0.0]
    ax3.scatter(no_exact_match['semantic_similarity'], no_exact_match['keyword_overlap'], 
                alpha=0.5, label='No Exact Match', color='orange')
    ax3.scatter(exact_match['semantic_similarity'], exact_match['keyword_overlap'], 
                alpha=0.7, label='Exact Match', color='green', marker='x', s=100)
    ax3.set_xlabel('Semantic Similarity')
    ax3.set_ylabel('Keyword Overlap')
    ax3.set_title('Exact Match vs Semantic Similarity')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 4. Score comparison
    ax4 = axes[1, 1]
    ax4.scatter(results_df['solution_score'], results_df['semantic_similarity'], alpha=0.5)
    ax4.set_xlabel('Ground Truth Solution Score (Reddit upvotes)')
    ax4.set_ylabel('Semantic Similarity')
    ax4.set_title('Solution Quality vs LLM Performance')
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


In [12]:
# Option 1: Extract problem/solution pairs using score-based strategy (default)
print("="*80)
print("EXTRACTING PAIRS WITH SCORE-BASED STRATEGY")
print("="*80)
pairs_df_score = extract_problem_solution_pairs(
    posts_df, 
    comments_df, 
    use_top_level_only=True,
    strategy="score_based"
)

# Display sample
print("\n=== SAMPLE PROBLEM/SOLUTION PAIRS (Score-Based) ===")
for idx, row in pairs_df_score.head(3).iterrows():
    print(f"\n{'='*80}")
    print(f"Post ID: {row['post_id']}")
    print(f"Problem (first 200 chars): {row['problem'][:200]}...")
    print(f"Solution: {row['solution']}")
    print(f"Solution Score: {row['solution_score']}")
    print(f"{'='*80}")

# Use score-based pairs for evaluation (faster for testing)
pairs_df = pairs_df_score


EXTRACTING PAIRS WITH SCORE-BASED STRATEGY
Filtered to 5691 solved self-posts
Valid comments: 345153
Using strategy: score_based


Extracting pairs: 100%|██████████| 5691/5691 [00:04<00:00, 1382.25it/s]


Extracted 5688 problem/solution pairs

=== SAMPLE PROBLEM/SOLUTION PAIRS (Score-Based) ===

Post ID: vs8rm
Problem (first 200 chars): Heard this one round the campfire years ago. Kept us stumped for a while; see how you do!

A police station received a call one day from an old lady, who had heard some shouting and a gunshot from a w...
Solution: He was the infamous bicycle thief/smuggler/kingpin of the city. He was meeting with the bicycle cartel of another area of the city to discuss a compromise on territory, but the other cartel decided to just end it there and kill the man.
Solution Score: 7

Post ID: w58oy
Problem (first 200 chars): Every number is 4.

Give me a number and I will show you how.  Figure out why it's true....
Solution: [You break down the number of letters in a given number as it is spelled out. If the result is a number other than four, you take that number and use its letters. Repeat as necessary.](/sp) [Seventeen is nine letters, nine is four letters. Five is fou

In [ ]:
# Option 1: Extract problem/solution pairs using score-based strategy
# sample_size = 50  # Start with 50 for testing
max_workers = 2  # Adjust based on API rate limits (5-20 recommended)

# print(f"Evaluating on {sample_size} sample riddles...")
print(f"Using {max_workers} parallel workers for faster processing")

# Evaluate with GPT-4o-mini (now with parallel processing)
results_df = evaluate_llm_on_riddles(
    pairs_df, 
    model="openai/gpt-4o-mini",
    save_results=True,
    max_workers=max_workers  # Add parallel processing!
)

Using 2 parallel workers for faster processing
Evaluating openai/gpt-4o-mini on 5688 riddles...
Using parallel processing with 2 workers...
Note: Adjust max_workers based on your API rate limits


Evaluating openai/gpt-4o-mini:   0%|          | 1/5688 [00:00<30:52,  3.07it/s]

Error calling LLM: Error code: 403 - {'error': {'message': 'Key limit exceeded (total limit). Manage it using https://openrouter.ai/settings/keys', 'code': 403}}
Error calling LLM: Error code: 403 - {'error': {'message': 'Key limit exceeded (total limit). Manage it using https://openrouter.ai/settings/keys', 'code': 403}}


Evaluating openai/gpt-4o-mini:   0%|          | 3/5688 [00:00<16:27,  5.76it/s]

Error calling LLM: Error code: 403 - {'error': {'message': 'Key limit exceeded (total limit). Manage it using https://openrouter.ai/settings/keys', 'code': 403}}
Error calling LLM: Error code: 403 - {'error': {'message': 'Key limit exceeded (total limit). Manage it using https://openrouter.ai/settings/keys', 'code': 403}}


Evaluating openai/gpt-4o-mini:   0%|          | 6/5688 [00:00<12:45,  7.42it/s]

... (238 outputs left out)

Error calling LLM: Error code: 403 - {'error': {'message': 'Key limit exceeded (total limit). Manage it using https://openrouter.ai/settings/keys', 'code': 403}}
Error calling LLM: Error code: 403 - {'error': {'message': 'Key limit exceeded (total limit). Manage it using https://openrouter.ai/settings/keys', 'code': 403}}


Evaluating openai/gpt-4o-mini:  11%|█         | 632/5688 [00:29<05:32, 15.22it/s]

Error calling LLM: Error code: 403 - {'error': {'message': 'Key limit exceeded (total limit). Manage it using https://openrouter.ai/settings/keys', 'code': 403}}
Error calling LLM: Error code: 403 - {'error': {'message': 'Key limit exceeded (total limit). Manage it using https://openrouter.ai/settings/keys', 'code': 403}}
Error calling LLM: Error code: 403 - {'error': {'message': 'Key limit exceeded (total limit). Manage it using https://openrouter.ai/settings/keys', 'code': 403}}
Error calling LLM: Error code: 403 - {'error': {'message': 'Key limit exceeded (total limit). Manage it using https://openrouter.ai/settings/keys', 'code': 403}}


Evaluating openai/gpt-4o-mini:  11%|█         | 634/5688 [00:29<05:47, 14.55it/s]

In [ ]:
# Analyze results
display_evaluation_summary(results_df)
display_sample_results(results_df, n=3, show_best=True)
display_sample_results(results_df, n=3, show_best=False)

In [ ]:
# Create visualizations
plot_evaluation_metrics(results_df)

In [ ]:
# Option 2: Extract with LLM strategy on full dataset (uncomment to run)
# WARNING: This will make many API calls and take a long time
# sample_size = 100
max_workers = 10  # Adjust based on API rate limits (5-20 recommended)

print("="*80)
print(f"EXTRACTING PAIRS WITH LLM-BASED STRATEGY (Parallel Processing)")
# print(f"Sample size: {sample_size}, Workers: {max_workers}")
print("="*80)

pairs_df_llm = extract_problem_solution_pairs(
    posts_df, 
    comments_df, 
    use_top_level_only=True,
    strategy="llm_based",
    llm_model="openai/gpt-4o-mini",
    max_workers=max_workers  # Add max_workers parameter for parallel processing
)

# Save LLM-based pairs
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = f"pairs_llm_based_{timestamp}.json"
pairs_df_llm.to_json(output_file, orient='records', indent=2)
print(f"\nLLM-based pairs saved to {output_file}")

# Use LLM-based pairs for evaluation
pairs_df = pairs_df_llm


EXTRACTING PAIRS WITH LLM-BASED STRATEGY (Parallel Processing)
Filtered to 5691 solved self-posts
Valid comments: 345153
Using strategy: llm_based
Using parallel processing with 10 workers...
Note: Adjust max_workers based on your API rate limits


Extracting pairs (parallel):  72%|███████▏  | 4113/5691 [04:40<01:34, 16.66it/s]

LLM error in select_best_answer_with_llm: Error code: 403 - {'error': {'message': 'Key limit exceeded (total limit). Manage it using https://openrouter.ai/settings/keys', 'code': 403}}
LLM error in select_best_answer_with_llm: Error code: 403 - {'error': {'message': 'Key limit exceeded (total limit). Manage it using https://openrouter.ai/settings/keys', 'code': 403}}


Extracting pairs (parallel):  72%|███████▏  | 4120/5691 [04:41<01:20, 19.54it/s]

LLM error in select_best_answer_with_llm: Error code: 403 - {'error': {'message': 'Key limit exceeded (total limit). Manage it using https://openrouter.ai/settings/keys', 'code': 403}}


Extracting pairs (parallel):  72%|███████▏  | 4123/5691 [04:41<01:29, 17.53it/s]

... (36 outputs left out)

LLM error in select_best_answer_with_llm: Error code: 403 - {'error': {'message': 'Key limit exceeded (total limit). Manage it using https://openrouter.ai/settings/keys', 'code': 403}}
LLM error in select_best_answer_with_llm: Error code: 403 - {'error': {'message': 'Key limit exceeded (total limit). Manage it using https://openrouter.ai/settings/keys', 'code': 403}}
LLM error in select_best_answer_with_llm: Error code: 403 - {'error': {'message': 'Key limit exceeded (total limit). Manage it using https://openrouter.ai/settings/keys', 'code': 403}}
LLM error in select_best_answer_with_llm: Error code: 403 - {'error': {'message': 'Key limit exceeded (total limit). Manage it using https://openrouter.ai/settings/keys', 'code': 403}}
LLM error in select_best_answer_with_llm: Error code: 403 - {'error': {'message': 'Key limit exceeded (total limit). Manage it using https://openrouter.ai/settings/keys', 'code': 403}}


Extracting pairs (parallel):  76%|███████▌  | 4300/5691 [04:46<01:00, 23.14it/s]

LLM error in select_best_answer_with_llm: Error code: 403 - {'error': {'message': 'Key limit exceeded (total limit). Manage it using https://openrouter.ai/settings/keys', 'code': 403}}
LLM error in select_best_answer_with_llm: Error code: 403 - {'error': {'message': 'Key limit exceeded (total limit). Manage it using https://openrouter.ai/settings/keys', 'code': 403}}
LLM error in select_best_answer_with_llm: Error code: 403 - {'error': {'message': 'Key limit exceeded (total limit). Manage it using https://openrouter.ai/settings/keys', 'code': 403}}
LLM error in select_best_answer_with_llm: Error code: 403 - {'error': {'message': 'Key limit exceeded (total limit). Manage it using https://openrouter.ai/settings/keys', 'code': 403}}
LLM error in select_best_answer_with_llm: Error code: 403 - {'error': {'message': 'Key limit exceeded (total limit). Manage it using https://openrouter.ai/settings/keys', 'code': 403}}
LLM error in select_best_answer_with_llm: Error code: 403 - {'error': {'mes

Extracting pairs (parallel):  76%|███████▌  | 4306/5691 [04:46<00:59, 23.23it/s]

In [ ]:
# Evaluate on a sample of riddles (now with parallel processing - much faster!)
# You can increase max_samples or remove it to evaluate on all pairs

# sample_size = 50  # Start with 50 for testing
max_workers = 10  # Adjust based on API rate limits (5-20 recommended)

# print(f"Evaluating on {sample_size} sample riddles...")
print(f"Using {max_workers} parallel workers for faster processing")

# Evaluate with GPT-4o-mini (now with parallel processing)
results_df = evaluate_llm_on_riddles(
    pairs_df, 
    model="openai/gpt-4o-mini",
    save_results=True,
    max_workers=max_workers  # Add parallel processing!
)

Using 10 parallel workers for faster processing
Evaluating openai/gpt-4o-mini on 5038 riddles...
Using parallel processing with 10 workers...
Note: Adjust max_workers based on your API rate limits


Evaluating openai/gpt-4o-mini:   0%|          | 16/5038 [00:03<17:40,  4.74it/s] 


In [ ]:
# Analyze results
display_evaluation_summary(results_df)
display_sample_results(results_df, n=3, show_best=True)
display_sample_results(results_df, n=3, show_best=False)

In [ ]:
# Create visualizations
plot_evaluation_metrics(results_df)